# RAG Basics: Retrieval-Augmented Generation

This notebook introduces Retrieval-Augmented Generation (RAG), which enhances LLM responses by retrieving relevant information from a knowledge base.

## Learning Objectives
- Understand the RAG architecture pattern
- Create embeddings from text documents
- Build a vector store for semantic search
- Combine retrieval with generation for context-aware responses

## Related Theoretical Content
- [RAG Systems](../../notes/03-ai/README.md)
- [Vector Databases](../../notes/03-ai/05-vector_databases/README.md)
- [Embeddings](../../notes/03-ai/04-embeddings/README.md)

## Prerequisites
This notebook is self-contained and can run independently.

## Setup: Import Dependencies and API Keys

Load required libraries and configure API access.

In [ ]:
import os
from api_keys import get_api_key

# Set environment variables
os.environ["OPENAI_API_KEY"] = get_api_key('openai')

print(" Environment configured")

## 1. Download Sample Documents

We'll create a simple knowledge base about cricket players using Wikipedia. This demonstrates how to:
1. Fetch content from external sources
2. Store it locally
3. Prepare it for embedding

In [ ]:
import wikipedia

# Famous cricket players to download
players = [
    "Sachin Tendulkar", "Jacques Kallis", "AB De Villiers",
    "Matthew Hayden", "Brian Lara", "Virat Kohli",
    "Kumar Sangakkara", "Mahela Jayawardene"
]

# Create data directory
data_dir = ".data/cricket_players"
os.makedirs(data_dir, exist_ok=True)

print("Downloading player biographies...")
for player in players:
    try:
        # Fetch Wikipedia summary
        page = wikipedia.page(player, auto_suggest=False)
        content = page.summary

        # Save to file
        filename = f"{player.replace(' ', '_')}.txt"
        filepath = os.path.join(data_dir, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)

        print(f" {player}")
    except Exception as e:
        print(f"✗ {player}: {e}")

print(f"\n Documents saved to {data_dir}")

## 2. Initialize Embedding Model

**Embeddings** convert text into numerical vectors that capture semantic meaning. Similar texts have similar vectors.

We use `sentence-transformers/all-MiniLM-L6-v2`:
- Fast and efficient
- Good for semantic similarity
- Runs locally (no API calls)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test the embedding
sample_embedding = embedding_model.embed_query("Who is Sachin Tendulkar?")
print(f" Embedding model loaded")
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"Sample values: {sample_embedding[:5]}...")

## 3. Load Documents into Memory

Load the text files and wrap them in LangChain's `Document` objects. Each document includes:
- **page_content**: The actual text
- **metadata**: Information about the document (source file, etc.)

In [ ]:
from langchain_core.documents import Document

docs = []

# Load all text files from the directory
for filename in os.listdir(data_dir):
    if filename.endswith(".txt"):
        filepath = os.path.join(data_dir, filename)

        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()

            # Create Document with metadata
            docs.append(Document(
                page_content=content,
                metadata={"source": filename}
            ))

print(f" Loaded {len(docs)} documents")
print(f"\nSample document:")
print(f"Source: {docs[0].metadata['source']}")
print(f"Content preview: {docs[0].page_content[:200]}...")

## 4. Create Vector Store

A **vector store** (or vector database) stores embeddings and enables fast semantic search.

**ChromaDB** is an open-source vector database that:
- Stores documents and their embeddings
- Performs similarity search
- Persists data to disk

In [ ]:
from langchain_chroma import Chroma

# Create vector store with persistence
vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory=".data/vector_db_cricket"
)

# Add documents (this creates embeddings automatically)
print("Creating embeddings and storing in vector database...")
vector_store.add_documents(docs)

print(f" Vector store created with {len(docs)} documents")

## 5. Semantic Search

Now we can search for relevant documents using natural language queries. The vector store:
1. Converts the query to an embedding
2. Finds documents with similar embeddings
3. Returns the k most relevant documents

In [ ]:
# Perform similarity search
user_query = "Who is Sachin?"
results = vector_store.similarity_search(user_query, k=2)

print(f"Query: '{user_query}'\n")
print(f"Found {len(results)} relevant documents:\n")

for i, result in enumerate(results, 1):
    print(f"{i}. Source: {result.metadata['source']}")
    print(f"   Content: {result.page_content[:200]}...")
    print()

## 6. RAG: Combine Retrieval with Generation

The full RAG pipeline:
1. **Retrieve**: Find relevant documents from vector store
2. **Augment**: Add retrieved context to the prompt
3. **Generate**: LLM generates response based on context

This allows the LLM to answer questions using specific knowledge not in its training data.

In [ ]:
from langchain_groq import ChatGroq

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=get_api_key('groq')
)

# User query
user_query = "What records did Sachin Tendulkar achieve in cricket?"

# Step 1: Retrieve relevant documents
relevant_docs = vector_store.similarity_search(user_query, k=2)

# Step 2: Combine retrieved content into context
context = "\n\n".join([doc.page_content for doc in relevant_docs])

# Step 3: Create prompt with context
prompt = f'''
You are an assistant tasked with answering user queries based on the provided context.
Generate the most accurate and concise answer based on the context alone.
If the context does not contain enough information, say "No relevant information found".

User Query: {user_query}

Context:
{context}
'''

# Step 4: Generate response
response = llm.invoke(prompt)

print(f"Question: {user_query}\n")
print(f"Answer: {response.content}")

## 7. Interactive RAG Query

Create a reusable function to query the RAG system.

In [ ]:
def query_rag(question: str, k: int = 3) -> str:
    """
    Query the RAG system with a question.

    Args:
        question: The user's question
        k: Number of relevant documents to retrieve

    Returns:
        The LLM's answer based on retrieved context
    """
    # Retrieve
    relevant_docs = vector_store.similarity_search(question, k=k)

    # Augment
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    prompt = f'''
    Answer the following question based only on the provided context.
    Be concise and accurate. If the context doesn't contain the answer, say so.

    Question: {question}

    Context:
    {context}
    '''

    # Generate
    response = llm.invoke(prompt)
    return response.content

# Test with multiple queries
questions = [
    "Who is Virat Kohli?",
    "Which player is known as the Master Blaster?",
    "What teams did Brian Lara play for?"
]

for question in questions:
    answer = query_rag(question)
    print(f"Q: {question}")
    print(f"A: {answer}\n")

## Key Takeaways

1. **RAG Architecture**: Retrieve → Augment → Generate
2. **Embeddings**: Convert text to vectors for semantic search
3. **Vector Stores**: Efficiently store and search embeddings
4. **Context Injection**: Provide LLM with relevant information
5. **Knowledge Grounding**: Answers based on specific documents, not just model knowledge

## Limitations of Basic RAG

- Long documents may exceed context limits
- Entire document is retrieved, even if only part is relevant
- No handling of multi-hop reasoning

## Next Steps

- [04-advanced-rag.ipynb](04-advanced-rag.ipynb): Document splitting, FAISS, and advanced techniques
- [05-agentic-ai.ipynb](05-agentic-ai.ipynb): Build AI agents with tools and decision-making